We have a speech-to-text notebook that:

* Processes multiple audio files
* Splits each file into overlapping chunks
* Transcribes the chunks
* Merges transcriptions for each file
* Exports the full transcription into a single `.docx` per file

Configure the parameters in the notebook and run it to generate transcriptions.

We don’t use standard GenAI pipelines (e.g., LangChain) since our focus is speech, not text—custom code is simpler and more efficient.

We use OpenAI s2t models for transcriptions.

In [24]:
from pathlib import Path
import ffmpeg
from tqdm import tqdm
from docx import Document
from openai import OpenAI
from paths import DATA_DIR, OUT_DIR

### API

In [ ]:
client = OpenAI()


def format_ts(s: float) -> str:
    s = max(0, int(s))
    m, s = divmod(s, 60)

    return f"{m:d}:{s:02d}"


def probe_duration(path: Path) -> float:
    return float(ffmpeg.probe(str(path))["format"]["duration"])


def clear_dir(d: Path) -> None:
    d.mkdir(parents=True, exist_ok=True)
    
    for f in d.glob("*"):
        if f.is_file():
            f.unlink()


def plan_base_chunks(total: float, chunk_len_sec: int) -> list[tuple[float, float]]:
    plan = []
    start = 0.0
    
    while start < total:
        end = min(start + chunk_len_sec, total)
        if end <= start:
            break
        plan.append((start, end))
        start += chunk_len_sec

    return plan


def extend_with_overlap(
    start: float, end: float, total: float, overlap_sec: int
) -> tuple[int, int]:
    rs = max(0.0, start - overlap_sec)
    re = min(total, end + overlap_sec)

    return int(rs), int(re)


def chunk_filename(idx1: int, real_start: int, real_end: int, suffix: str) -> str:
    return f"chunk_{idx1:03d}_{real_start:06d}-{real_end:06d}{suffix}"


def write_chunk(src: Path, dst: Path, real_start: int, real_end: int) -> None:
    (
        ffmpeg.input(str(src), ss=real_start, t=real_end - real_start)
        .output(str(dst), c="copy")
        .overwrite_output()
        .run(quiet=True)
    )


def split_audio_with_overlap(
    path: Path, outdir: Path, chunk_len: int = 300, overlap: int = 30
) -> list[tuple[Path, int, int]]:
    clear_dir(outdir)
    total = probe_duration(path)
    base = plan_base_chunks(total, chunk_len)
    outputs: list[tuple[Path, int, int]] = []

    for i, (bs, be) in enumerate(base, start=1):
        rs, re = extend_with_overlap(bs, be, total, overlap)
    
        if re <= rs:
            continue
    
        outpath = outdir / chunk_filename(i, rs, re, path.suffix)
        write_chunk(path, outpath, rs, re)
        outputs.append((outpath, rs, re))
    
    return outputs


def transcribe_chunk(
    p: Path,
    model: str = "gpt-4o-transcribe",
    prompt: str | None = None,
    language: str | None = None,
    temperature: float = 0,
) -> str:
    with open(p, "rb") as f:
        r = client.audio.transcriptions.create(
            model=model,
            file=f,
            prompt=prompt,
            language=language,
            temperature=temperature,
        )

    return r.text


def transcribe_chunks(
    chunks: list[tuple[Path, int, int]],
    model: str,
    prompt: str | None,
    language: str | None,
) -> list[tuple[str, int, int, str]]:
    results: list[tuple[str, int, int, str]] = []

    for outpath, rs, re in tqdm(chunks, desc="Transcribing", unit="chunk"):
        txt = transcribe_chunk(outpath, model=model, prompt=prompt, language=language)
        results.append((outpath.name, rs, re, txt))

    return results


def build_docx(docx_path: Path, results: list[tuple[str, int, int, str]]) -> None:
    if docx_path.exists():
        docx_path.unlink()
    
    doc = Document()

    for idx, (_fname, rs, re, txt) in enumerate(results, start=1):
        p = doc.add_paragraph()
        run = p.add_run(f"Chunk {idx} ({format_ts(rs)}–{format_ts(re)})")
        run.bold = True
        doc.add_paragraph(txt or "")

    doc.save(str(docx_path))


def process_file(
    path: Path,
    out_root: Path,
    chunk_len_sec: int = 300,
    overlap_sec: int = 30,
    model: str = "gpt-4o-transcribe",
    prompt: str | None = None,
    language: str | None = None,
) -> Path:
    rel = path.parent.name
    chunk_dir = out_root / rel / f"chunks_{path.name}"
    text_dir = out_root / rel / "text"
    text_dir.mkdir(parents=True, exist_ok=True)

    chunks = split_audio_with_overlap(
        path, chunk_dir, chunk_len=chunk_len_sec, overlap=overlap_sec
    )
    results = transcribe_chunks(chunks, model=model, prompt=prompt, language=language)

    docx_path = text_dir / f"chunks_{path.stem}_cl{chunk_len_sec}_ol{overlap_sec}.docx"
    build_docx(docx_path, results)

    return docx_path


def process_files(
    paths: list[Path],
    out_root: Path,
    chunk_len_sec: int = 300,
    overlap_sec: int = 30,
    model: str = "gpt-4o-transcribe",
    prompt: str | None = None,
    language: str | None = None,
) -> list[Path]:
    return [
        process_file(
            p,
            out_root,
            chunk_len_sec=chunk_len_sec,
            overlap_sec=overlap_sec,
            model=model,
            prompt=prompt,
            language=language,
        )
        for p in paths
    ]


#### Parameters

In [ ]:
FILES_DIR = DATA_DIR / "25-09-12"

FILES_EXT = ".MP3"

CHUNK_LEN_SEC = 30

OVERLAP_SEC = 1

PROMPT = """Recordings of an interview or event at the Lithuanian National Opera.
Speakers often switch between Lithuanian and English.
They may have unclear accents, but the language is always either Lithuanian or English.
There may be background noise or remarks from the audience, but only the speakers matter."""

MODEL = "gpt-4o-transcribe"

#### Run

In [ ]:
FILES = list(FILES_DIR).glob(f"*.{FILES_EXT}")

process_files(
    paths=FILES,
    out_root=OUT_DIR,
    chunk_len_sec=CHUNK_LEN_SEC,
    overlap_sec=OVERLAP_SEC,
    model=MODEL,
    prompt=PROMPT
)

Transcribing 201212_0601.MP3: 100%|██████████| 10/10 [03:15<00:00, 19.59s/chunk]


[PosixPath('/home/tufman/src/personal/s2t/out/25-09-12/text/chunks_201211_0597_cl300_ol1.docx'),
 PosixPath('/home/tufman/src/personal/s2t/out/25-09-12/text/chunks_201211_0598_cl300_ol1.docx'),
 PosixPath('/home/tufman/src/personal/s2t/out/25-09-12/text/chunks_201212_0599_cl300_ol1.docx'),
 PosixPath('/home/tufman/src/personal/s2t/out/25-09-12/text/chunks_201212_0600_cl300_ol1.docx'),
 PosixPath('/home/tufman/src/personal/s2t/out/25-09-12/text/chunks_201212_0601_cl300_ol1.docx')]